# Week 05 — Function 04

## Table of contents

1. [Overview](#overview)
2. [Objectives](#objectives)
3. [Evidence provenance](#evidence-provenance)
4. [Environment and setup](#environment-setup)
5. [Data validation](#data-validation)
6. [Descriptive EDA](#descriptive-eda)
7. [Visual EDA](#visual-eda)
8. [Model and acquisition](#model-acquisition)
9. [Week 5 proposal](#week-5-proposal)
10. [Reproducibility checks](#reproducibility-checks)
11. [Conclusions and next steps](#conclusions-next-steps)

<a id="overview"></a>
## 1. Overview

This focused review mirrors the canonical Week 5 methodology for Function 04, with Weeks 1–4 observed and Week 5 proposed only.

<a id="objectives"></a>
## 2. Objectives

Validate the 4-dimensional evidence, assess the latest returned point, and reproduce the recorded GP-UCB proposal without look-ahead.

<a id="evidence-provenance"></a>
## 3. Evidence provenance

Starter arrays come from `Week_01/Function_nn/03_Data`; exact returned pairs come from `Results/query_output_ledger.csv`. The Week 5 return is excluded because it was unknown when the proposal was selected.

<a id="environment-setup"></a>
## 4. Environment and setup

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT=Path.cwd().resolve()
for candidate in (ROOT,*ROOT.parents):
    if (candidate/'Week_05'/'Function_04').is_dir(): ROOT=candidate; break
else: raise FileNotFoundError('Could not locate repository root')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from Code.historical_function_review import analyse_historical_function

<a id="data-validation"></a>
## 5. Data validation

In [2]:
observations, summary, proposal, diagnostic_figure = analyse_historical_function(5, 4, ROOT)
input_columns=[f'x{i}' for i in range(1,5)]
inputs=observations[input_columns].to_numpy(float)
outputs=observations['objective'].to_numpy(float)
assert inputs.shape==(34,4) and outputs.shape==(34,)
assert np.isfinite(inputs).all() and np.isfinite(outputs).all()
assert np.all((inputs>=0)&(inputs<=1))
observations

,query,evidence,x1,x2,x3,x4,objective
0,1,starter,0.896981,0.725628,0.175404,0.701694,-22.108288
1,2,starter,0.889356,0.499588,0.539269,0.508783,-14.601397
2,3,starter,0.250946,0.033693,0.145380,0.494932,-11.699932
3,4,starter,0.346962,0.006250,0.760564,0.613024,-16.053765
4,5,starter,0.124871,0.129770,0.384400,0.287076,-10.069633
5,6,starter,0.801303,0.500231,0.706645,0.195103,-15.487083
6,7,starter,0.247708,0.060445,0.042186,0.441324,-12.681685
7,8,starter,0.746702,0.757092,0.369353,0.206566,-16.026400
8,9,starter,0.400665,0.072574,0.886768,0.243842,-17.049235
9,10,starter,0.626071,0.586751,0.438806,0.778858,-12.741766


<a id="descriptive-eda"></a>
## 6. Descriptive EDA

All comparisons are descriptive and within-function; no causal, global-optimum, or cross-function ranking claim is made.

In [3]:
pd.Series({k:v for k,v in summary.items() if k!='proposal'}, name='verified evidence')

week                                                                             5
function                                                                         4
dimensions                                                                       4
starter_observations                                                            30
recorded_pairs                                                                   4
total_verified_observations                                                     34
best_query                                                                      33
best_input                                [0.394519, 0.361122, 0.256803, 0.461856]
best_output                                                              -1.981075
latest_verified_query                                                           34
latest_verified_input                     [0.715715, 0.504442, 0.275065, 0.552962]
latest_verified_output                                                   -9.312812
late

<a id="visual-eda"></a>
## 7. Visual EDA

Orange markers are returned Weeks 1–4 observations; the star is the verified incumbent. The Week 5 proposal is deliberately absent.

In [4]:
display(diagnostic_figure)
plt.close(diagnostic_figure)

<Figure size 1200x450 with 3 Axes>

<a id="model-acquisition"></a>
## 8. Model and acquisition

Method: **GP-UCB**. This adaptive policy is a heuristic chosen from the evidence available at the decision boundary; it is not a statistically controlled acquisition comparison.

<a id="week-5-proposal"></a>
## 9. Week 5 proposal

Proposed only: `[0.713766, 0.198601, 0.014095, 0.068347]`. Decision record: Chosen using evidence through Week 4, before the Week 5 return.

<a id="reproducibility-checks"></a>
## 10. Reproducibility checks

In [5]:
candidate=np.asarray(proposal['query'],dtype=float)
assert proposal['status']=='proposed_only'
assert candidate.shape==(4,) and np.all((candidate>=0)&(candidate<=0.999999))
duplicate=bool(np.any(np.all(np.isclose(inputs,candidate,rtol=0,atol=5e-7),axis=1)))
assert duplicate==proposal['duplicates_observed_evidence']
assert summary['recorded_pairs']==4
portal='-'.join(f'{value:.6f}' for value in candidate)
assert all(len(part.split('.')[-1])==6 for part in portal.split('-'))
print('Function 04 Week 5 checks passed:', portal, 'duplicate:', duplicate)

Function 04 Week 5 checks passed: 0.713766-0.198601-0.014095-0.068347 duplicate: False


<a id="conclusions-next-steps"></a>
## 11. Conclusions and next steps

The evidence boundary is locked at 34 verified observations. The Week 5 proposal remains unobserved until its authoritative return is appended at the next checkpoint.